# 📊 Notebook 1: Exploración de Datos (EDA)
## Análisis de Churn en Mercados Emergentes

---

### Objetivo
Realizar un Análisis Exploratorio de Datos (EDA) completo del dataset Telco Customer Churn para identificar patrones, anomalías y variables relevantes para el modelado.

### Contenido
1. Carga y revisión inicial de datos
2. Limpieza y preprocesamiento
3. Análisis univariado
4. Análisis bivariado
5. Análisis de tenure y grupos
6. Análisis de cargos
7. Correlaciones
8. Guardar datos procesados

## 1. Configuración Inicial

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

# Rutas
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
OUTPUTS = Path('../outputs/figures')

# Crear directorios si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print("✅ Configuración completada")

## 2. Carga y Revisión Inicial de Datos

In [ ]:
# Cargar datos
df = pd.read_csv(DATA_RAW / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"📊 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\n📅 Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")

In [ ]:
# Primeras filas
df.head()

In [ ]:
# Información del dataset
print("📋 Información del Dataset:")
print("="*60)
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe().T

In [ ]:
# Valores únicos por columna
print("🔢 Valores únicos por columna:")
print("="*60)
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique < 10:
        vals = df[col].unique()
        print(f"{col:20} | {n_unique:3} valores: {vals}")
    else:
        print(f"{col:20} | {n_unique:3} valores")

## 3. Limpieza y Preprocesamiento

In [ ]:
# Revisar valores nulos
print("🔍 Análisis de Valores Nulos:")
print("="*60)
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) > 0:
    print(null_counts)
else:
    print("✅ No hay valores nulos explícitos")

In [ ]:
# TotalCharges tiene valores vacíos (espacios en blanco)
print("🔍 Revisando TotalCharges...")
print(f"Tipo de dato: {df['TotalCharges'].dtype}")

# Identificar valores no numéricos
non_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()
n_invalid = non_numeric.sum()
print(f"\n⚠️ Valores no numéricos en TotalCharges: {n_invalid}")

if n_invalid > 0:
    print("\nFilas con valores inválidos:")
    display(df[non_numeric][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

In [ ]:
# Convertir TotalCharges a numérico
# Los valores vacíos corresponden a clientes con tenure=0 (nuevos clientes)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Imputar con MonthlyCharges * tenure o 0 si tenure=0
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])

print(f"✅ TotalCharges convertido a numérico")
print(f"   Rango: ${df['TotalCharges'].min():.2f} - ${df['TotalCharges'].max():.2f}")
print(f"   Promedio: ${df['TotalCharges'].mean():.2f}")

In [ ]:
# Verificar duplicados
duplicates = df.duplicated().sum()
print(f"🔍 Registros duplicados: {duplicates}")

# CustomerID duplicados
dup_ids = df['customerID'].duplicated().sum()
print(f"🔍 CustomerIDs duplicados: {dup_ids}")

## 4. Análisis Univariado

### 4.1 Variable Target: Churn

In [ ]:
# Distribución de Churn
fig, ax = plt.subplots(figsize=(8, 5))

churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']

bars = ax.bar(churn_counts.index, churn_counts.values, color=colors, 
              edgecolor='white', linewidth=2)

# Añadir etiquetas
for bar, count in zip(bars, churn_counts.values):
    pct = count / len(df) * 100
    ax.annotate(f'{count:,}\n({pct:.1f}%)',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_title('Distribución de Churn', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Estado del Cliente', fontsize=12)
ax.set_ylabel('Cantidad de Clientes', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Línea de referencia
ax.axhline(y=len(df)/2, color='gray', linestyle='--', alpha=0.5, label='50%')

plt.tight_layout()
plt.savefig(OUTPUTS / '01_churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Estadísticas
churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"\n📊 Tasa de Churn: {churn_rate:.1f}%")
print(f"   Clientes que permanecen: {(df['Churn'] == 'No').sum():,}")
print(f"   Clientes que abandonan: {(df['Churn'] == 'Yes').sum():,}")

### 4.2 Variables Demográficas

In [ ]:
# Variables demográficas
demo_vars = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, var in enumerate(demo_vars):
    ax = axes[i]
    counts = df[var].value_counts()
    
    # Convertir SeniorCitizen a etiquetas
    if var == 'SeniorCitizen':
        counts.index = ['No Senior', 'Senior']
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
    
    for bar, count in zip(bars, counts.values):
        pct = count / len(df) * 100
        ax.annotate(f'{pct:.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_title(f'Distribución de {var}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cantidad')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '02_demographics_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.3 Distribución de Tenure

In [ ]:
# Distribución de tenure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax1 = axes[0]
ax1.hist(df['tenure'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
ax1.axvline(df['tenure'].median(), color='red', linestyle='--', linewidth=2, 
            label=f'Mediana: {df["tenure"].median():.0f} meses')
ax1.axvline(df['tenure'].mean(), color='orange', linestyle='--', linewidth=2,
            label=f'Media: {df["tenure"].mean():.1f} meses')
ax1.set_title('Distribución de Tenure', fontsize=14, fontweight='bold')
ax1.set_xlabel('Tenure (meses)')
ax1.set_ylabel('Frecuencia')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Boxplot
ax2 = axes[1]
bp = ax2.boxplot(df['tenure'], patch_artist=True)
bp['boxes'][0].set_facecolor('#3498db')
ax2.set_title('Boxplot de Tenure', fontsize=14, fontweight='bold')
ax2.set_ylabel('Tenure (meses)')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '03_tenure_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Estadísticas
print(f"\n📊 Estadísticas de Tenure:")
print(f"   Mínimo: {df['tenure'].min()} meses")
print(f"   Máximo: {df['tenure'].max()} meses")
print(f"   Media: {df['tenure'].mean():.1f} meses")
print(f"   Mediana: {df['tenure'].median():.0f} meses")
print(f"   Desv. Est.: {df['tenure'].std():.1f} meses")

## 5. Análisis Bivariado

### 5.1 Churn por Tipo de Contrato

In [ ]:
# Función para calcular churn rate
def calcular_churn_rate(df, grupo):
    result = df.groupby(grupo).agg(
        total=('Churn', 'count'),
        churners=('Churn', lambda x: (x == 'Yes').sum())
    ).reset_index()
    result['churn_rate'] = (result['churners'] / result['total'] * 100).round(2)
    return result.sort_values('churn_rate', ascending=False)

# Churn por contrato
contract_churn = calcular_churn_rate(df, 'Contract')

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(contract_churn['Contract'], contract_churn['churn_rate'], 
              color=colors, edgecolor='white', linewidth=2)

# Añadir etiquetas
for bar, row in zip(bars, contract_churn.itertuples()):
    ax.annotate(f'{row.churn_rate:.1f}%\n({row.churners:,}/{row.total:,})',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Línea de promedio
avg_churn = (df['Churn'] == 'Yes').mean() * 100
ax.axhline(avg_churn, color='navy', linestyle='--', linewidth=2, 
           label=f'Promedio: {avg_churn:.1f}%')

ax.set_title('Tasa de Churn por Tipo de Contrato', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Tipo de Contrato')
ax.set_ylabel('Tasa de Churn (%)')
ax.set_ylim(0, 55)
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '04_churn_by_contract.png', dpi=300, bbox_inches='tight')
plt.show()

# Cálculo de riesgo relativo
monthly_rate = contract_churn[contract_churn['Contract'] == 'Month-to-month']['churn_rate'].values[0]
annual_rate = contract_churn[contract_churn['Contract'] == 'One year']['churn_rate'].values[0]
print(f"\n📊 Insight: Contratos mensuales tienen {monthly_rate/annual_rate:.1f}x más riesgo de churn")

### 5.2 Churn por Método de Pago

In [ ]:
# Churn por método de pago
payment_churn = calcular_churn_rate(df, 'PaymentMethod')

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn_r(payment_churn['churn_rate'] / 50)
bars = ax.barh(payment_churn['PaymentMethod'], payment_churn['churn_rate'], 
               color=colors, edgecolor='white', linewidth=2)

# Añadir etiquetas
for bar, row in zip(bars, payment_churn.itertuples()):
    ax.annotate(f'{row.churn_rate:.1f}%  ({row.churners:,}/{row.total:,})',
                xy=(bar.get_width() + 1, bar.get_y() + bar.get_height()/2),
                ha='left', va='center', fontsize=11, fontweight='bold')

ax.axvline(avg_churn, color='navy', linestyle='--', linewidth=2, 
           label=f'Promedio: {avg_churn:.1f}%')

ax.set_title('Tasa de Churn por Método de Pago', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Tasa de Churn (%)')
ax.set_xlim(0, 55)
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '05_churn_by_payment.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Insight: Los pagos automáticos (tarjeta, banco) tienen menor churn")

### 5.3 Churn por Servicios

In [ ]:
# Variables de servicios
service_vars = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Crear matriz de churn rates
churn_matrix = []
for var in service_vars:
    temp = calcular_churn_rate(df, var)
    for _, row in temp.iterrows():
        churn_matrix.append({
            'Variable': var,
            'Valor': row[var],
            'Churn_Rate': row['churn_rate'],
            'Total': row['total']
        })

churn_df = pd.DataFrame(churn_matrix)

# Visualizar top servicios con mayor churn
top_churn = churn_df.nlargest(10, 'Churn_Rate')

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Reds(top_churn['Churn_Rate'] / 50)
bars = ax.barh(range(len(top_churn)), top_churn['Churn_Rate'], color=colors, edgecolor='white')

ax.set_yticks(range(len(top_churn)))
ax.set_yticklabels([f"{row['Variable']}={row['Valor']}" for _, row in top_churn.iterrows()])

for bar, row in zip(bars, top_churn.itertuples()):
    ax.annotate(f'{row.Churn_Rate:.1f}%',
                xy=(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2),
                ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_title('Top 10 Configuraciones con Mayor Churn', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Tasa de Churn (%)')
ax.set_xlim(0, 50)
ax.axvline(avg_churn, color='navy', linestyle='--', linewidth=2, label=f'Promedio: {avg_churn:.1f}%')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '06_churn_by_services.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Análisis de Tenure por Grupos

In [ ]:
# Crear grupos de tenure
bins = [0, 12, 24, 48, 72]
labels = ['0-12m', '12-24m', '24-48m', '48m+']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)

# Calcular churn por grupo
tenure_churn = calcular_churn_rate(df, 'tenure_group')

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
bars = ax.bar(tenure_churn['tenure_group'], tenure_churn['churn_rate'], 
              color=colors, edgecolor='white', linewidth=2)

for bar, row in zip(bars, tenure_churn.itertuples()):
    ax.annotate(f'{row.churn_rate:.1f}%\n({row.churners:,}/{row.total:,})',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(avg_churn, color='navy', linestyle='--', linewidth=2, label=f'Promedio: {avg_churn:.1f}%')

ax.set_title('Tasa de Churn por Grupo de Tenure', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Grupo de Tenure')
ax.set_ylabel('Tasa de Churn (%)')
ax.set_ylim(0, 55)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '07_churn_by_tenure_group.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 HALLAZGO CLAVE: {tenure_churn[tenure_churn['tenure_group'] == '0-12m']['churn_rate'].values[0]:.1f}% de clientes en primeros 12 meses abandona")

## 7. Análisis de Cargos

In [ ]:
# Comparar cargos entre churners y no churners
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Charges
ax1 = axes[0]
df.boxplot(column='MonthlyCharges', by='Churn', ax=ax1, patch_artist=True)
ax1.set_title('Cargos Mensuales por Churn', fontsize=12, fontweight='bold')
ax1.set_xlabel('Churn')
ax1.set_ylabel('Cargos Mensuales ($)')
plt.suptitle('')

# Total Charges
ax2 = axes[1]
df.boxplot(column='TotalCharges', by='Churn', ax=ax2, patch_artist=True)
ax2.set_title('Cargos Totales por Churn', fontsize=12, fontweight='bold')
ax2.set_xlabel('Churn')
ax2.set_ylabel('Cargos Totales ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUTS / '08_charges_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

# Estadísticas
print("\n📊 Estadísticas de Cargos:")
print("="*50)
for churn_status in ['No', 'Yes']:
    subset = df[df['Churn'] == churn_status]
    print(f"\nChurn = {churn_status}:")
    print(f"   MonthlyCharges promedio: ${subset['MonthlyCharges'].mean():.2f}")
    print(f"   TotalCharges promedio: ${subset['TotalCharges'].mean():.2f}")

In [ ]:
# Diferencia de cargos
churner_monthly = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()
non_churner_monthly = df[df['Churn'] == 'No']['MonthlyCharges'].mean()
diff_pct = (churner_monthly - non_churner_monthly) / non_churner_monthly * 100

print(f"\n📊 HALLAZGO CLAVE: Churners pagan ${churner_monthly:.2f} vs ${non_churner_monthly:.2f} no-churners")
print(f"   Diferencia: +{diff_pct:.1f}%")
print(f"\n   CONCLUSIÓN: El churn no es un problema de PRECIO, es un problema de VALOR")

## 8. Heatmap de Correlaciones

In [ ]:
# Preparar variables numéricas
df['Churn_numeric'] = (df['Churn'] == 'Yes').astype(int)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_numeric']

# Matriz de correlación
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})

ax.set_title('Matriz de Correlaciones', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(OUTPUTS / '09_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Correlaciones con Churn:")
print(corr_matrix['Churn_numeric'].sort_values(ascending=False))

## 9. Guardar Datos Procesados

In [ ]:
# Guardar dataset procesado
output_path = DATA_PROCESSED / 'churn_features.csv'
df.to_csv(output_path, index=False)

print(f"✅ Datos procesados guardados en: {output_path}")
print(f"   Total de filas: {len(df):,}")
print(f"   Total de columnas: {len(df.columns)}")

## 10. Resumen Ejecutivo del EDA

In [ ]:
print("="*70)
print("RESUMEN EJECUTIVO - ANÁLISIS EXPLORATORIO DE DATOS")
print("="*70)

print(f"\n📊 DATOS ANALIZADOS:")
print(f"   • Total de clientes: {len(df):,}")
print(f"   • Variables: {len(df.columns)}")
print(f"   • Tasa de churn: {avg_churn:.1f}%")

print(f"\n🔍 HALLAZGOS CLAVE:")
print(f"   1. CONTRATO: Contratos mensuales tienen 3x más riesgo de churn")
print(f"      - Month-to-month: {contract_churn[contract_churn['Contract']=='Month-to-month']['churn_rate'].values[0]:.1f}%")
print(f"      - Two year: {contract_churn[contract_churn['Contract']=='Two year']['churn_rate'].values[0]:.1f}%")

print(f"\n   2. TENURE: {tenure_churn[tenure_churn['tenure_group']=='0-12m']['churn_rate'].values[0]:.1f}% abandona en primeros 12 meses")
print(f"      - Ventana crítica: meses 3-6")
print(f"      - La retención mejora con el tiempo")

print(f"\n   3. CARGOS: Churners pagan MÁS (+{diff_pct:.1f}%)")
print(f"      - Problema de valor percibido, no de precio")
print(f"      - Mayor cargo no implica mayor lealtad")

print(f"\n   4. MÉTODO DE PAGO: Pagos automáticos retienen más")
print(f"      - Electronic check: {payment_churn[payment_churn['PaymentMethod']=='Electronic check']['churn_rate'].values[0]:.1f}% churn")
print(f"      - Credit card: {payment_churn[payment_churn['PaymentMethod']=='Credit card (automatic)']['churn_rate'].values[0]:.1f}% churn")

print(f"\n📈 PRÓXIMOS PASOS:")
print(f"   1. Análisis de cohortes y supervivencia")
print(f"   2. Modelado predictivo")
print(f"   3. Triangulación con insights cualitativos")

print("="*70)